# KLEM Validation Notebook

End-to-end validation of the KLEM macroeconomic module within GHIM.

**Model structure:** GHIM uses a Cobb-Douglas production function $Y = A \cdot K^\alpha \cdot L^{1-\alpha}$ with TFP ($A$) calibrated from SSP GDP paths, and energy cost feedback that reduces net output, investment, and future capital. Technologies compete via preference-factor logit with stock turnover and learning curves.

**Data sources:**
- **SSP socioeconomic data** (`ghim/data/external/ssp/SSP_database_2024.csv.gz`): Population and GDP|PPP from the IIASA SSP database v3.0.1. Historical Reference scenario provides observed 2000-2015 values; SSP scenarios provide 2020-2100 projections.
- **Region mapping** (`ghim/data/external/region_classification.tsv`): ISO → AR6 R10 direct mapping (10 world regions).
- **Energy calibration** (`ghim/data/energy_cal.py`): Default base-year (2020) energy balance based on approximate IEA data.

**Purpose:** Validate KLEM initialization, TFP calibration (including historical K back-solve), single-period mechanics, and GDP fidelity across all 10 regions and 31 periods.

**Sections:**
1. SSP Data Preparation & Inspection
2. Energy Calibration Data Inspection
3. KLEM Initialization Validation
4. TFP Trajectory & Historical K Back-Solve
5. Single-Period KLEM Walkthrough
6. Full Model Run & Results Comparison
7. Diagnostic Checks & Summary

In [3]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({"figure.figsize": (14, 6), "font.size": 11})

from ghim.data.ssp import load_ssp_data
from ghim.data.energy_cal import (
    DEFAULT_FINAL_DEMAND, DEFAULT_ELEC_TOTAL_EJ,
    DEFAULT_ELEC_SHARES, DEFAULT_PRIMARY_ENERGY,
)
from ghim.econ.klem import KLEMDriver
from ghim.config import (
    BASE_YEAR, MODEL_YEARS, HISTORICAL_YEARS, FUTURE_YEARS, TIMESTEP,
    CAPITAL_OUTPUT_RATIO, LABOR_FORCE_PARTICIPATION, CAPITAL_SHARE,
    SAVINGS_RATE, INVESTMENT_CAP_RATE, DEPRECIATION_RATE,
)
from ghim.regions import R10_REGIONS
from ghim.solver.recursive import run_model, build_region_model
from ghim.output.reporting import results_to_dataframe

print(f"Python {sys.version}")
print(f"Regions: {len(R10_REGIONS)}, Periods: {len(MODEL_YEARS)} ({MODEL_YEARS[0]}-{MODEL_YEARS[-1]})")
print(f"Base year: {BASE_YEAR}, Timestep: {TIMESTEP} yr")

Python 3.11.14 (main, Oct 21 2025, 18:31:21) [GCC 11.2.0]
Regions: 10, Periods: 31 (2000-2150)
Base year: 2020, Timestep: 5 yr


---
## 1. SSP Data Preparation & Inspection

In [4]:
ssp_data = load_ssp_data("SSP2")
pop_df = ssp_data["population"]
gdp_df = ssp_data["gdp"]

print(f"Population shape: {pop_df.shape}")
print(f"GDP shape:        {gdp_df.shape}")
print(f"Regions: {list(pop_df.index)}")
print(f"Year range: {pop_df.columns.min()} - {pop_df.columns.max()}")

Population shape: (10, 31)
GDP shape:        (10, 31)
Regions: ['Africa', 'Asia-Pacific Developed', 'Eastern Asia', 'Eurasia', 'Europe', 'Latin America and Caribbean', 'Middle East', 'North America', 'South-East Asia and developing Pacific', 'Southern Asia']
Year range: 2000 - 2150


### EDIT: Add source for region mappings: https://pure.iiasa.ac.at/id/eprint/19306/ Smith, C., Byers, E. , Werning, M., & Hooke, D. (2023). R10 region mask based on IPCC AR6 WG3 and ISIMIP. 10.5281/zenodo.7629799.
### EDIT: And where's this SSP2 data come from? You can at leaset tell the source location in the project directory
### QUESTION: Shall we go with PPP or MER. GCAM goes with MER but are there any models go with PPP? Why some models go with PPP and others go with MER. I mean what is the core difference?

**Data sources:**
- **Region mapping:** AR6 R10 regions from Smith, C., Byers, E., Werning, M., & Hooke, D. (2023). *R10 region mask based on IPCC AR6 WG3 and ISIMIP.* [doi:10.5281/zenodo.7629799](https://doi.org/10.5281/zenodo.7629799). Stored at `ghim/data/external/region_classification.tsv`.
- **SSP data:** IIASA SSP database v3.0.1 (`ghim/data/external/ssp/SSP_database_2024.csv.gz`). Historical Reference scenario for 2000-2015 observed values; SSP2 scenario for 2020-2100 projections. Extrapolated to 2150.

**GDP measure — PPP vs MER:**

GHIM uses **GDP|PPP** (Purchasing Power Parity), not MER (Market Exchange Rates). This differs from GCAM (which uses MER).

| | PPP | MER |
|---|---|---|
| **Used by** | DICE, REMIND, WITCH, MESSAGE, SSP database default | GCAM, some IPCC scenarios |
| **Advantage** | Better reflects real living standards and energy demand drivers; more stable across time | Consistent with trade flows and international finance |
| **Effect on developing regions** | Higher GDP (corrects for lower local prices) | Lower GDP (reflects weaker currencies) |

PPP is preferred for energy modeling because energy demand correlates with real economic activity (purchasing power), not nominal exchange rates. The SSP database provides both, but most IAMs use PPP for energy/emissions projections. MER can understate developing-region GDP by 2-4x, distorting energy demand drivers.

In [6]:
### EDIT(?): The graphas for pop and gdp are fine but is not intuitive enough? Can we just have cool scientific line graphs? What do you say?

### I didn't see this graph before. Cool but isn't there any data before 2015?

In [ ]:
# Base-year summary table
base_summary = pd.DataFrame({
    "Population (M)": pop_df[BASE_YEAR],
    "GDP (B USD)": gdp_df[BASE_YEAR],
    "GDP/cap (K USD)": gdp_df[BASE_YEAR] / pop_df[BASE_YEAR] * 1000,
}).round(1)

print(f"\nBase-year ({BASE_YEAR}) summary:")
display(base_summary)

**Note:** Historical data (2000-2015) comes from the "Historical Reference" scenario in the SSP database, which provides real observed GDP and population. SSP scenario projections begin from 2020. The graphs above should show distinct historical values from 2000 onward, not flat lines copied from 2020.

In [ ]:
# Final demand table (region x sector)
fd_df = pd.DataFrame(DEFAULT_FINAL_DEMAND).T
fd_df["total"] = fd_df.sum(axis=1)
fd_df = fd_df.reindex(R10_REGIONS)

print("Default Final Energy Demand (EJ):")
display(fd_df)

In [ ]:
# Electricity total by region (bar chart)
elec_total = pd.Series(DEFAULT_ELEC_TOTAL_EJ).reindex(R10_REGIONS)

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

elec_total.plot.bar(ax=axes[0], color="steelblue")
axes[0].set_title("Electricity Generation by Region (EJ)")
axes[0].set_ylabel("EJ")
axes[0].tick_params(axis="x", rotation=45)

# Stacked bar chart of electricity shares
shares_df = pd.DataFrame(DEFAULT_ELEC_SHARES).T.reindex(R10_REGIONS).fillna(0)
shares_df.plot.bar(stacked=True, ax=axes[1], colormap="tab10")
axes[1].set_title("Electricity Generation Shares by Region")
axes[1].set_ylabel("Share")
axes[1].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

In [ ]:
# Primary energy table (region x fuel)
pe_df = pd.DataFrame(DEFAULT_PRIMARY_ENERGY).T.reindex(R10_REGIONS).fillna(0)
pe_df["total"] = pe_df.sum(axis=1)

print("Default Primary Energy (EJ):")
display(pe_df.round(2))

---
## 3. KLEM Initialization Validation

In [ ]:
# Build KLEMDriver for each region and validate initialization
init_rows = []
init_checks = []

for region in R10_REGIONS:
    base_gdp = float(gdp_df.loc[region, BASE_YEAR])
    base_pop = float(pop_df.loc[region, BASE_YEAR])
    fd = DEFAULT_FINAL_DEMAND.get(region, {"industry": 5.0, "buildings": 5.0, "transport": 5.0})
    total_final = sum(fd.values())

    klem = KLEMDriver(base_gdp, base_pop, total_final)

    init_rows.append({
        "region": region,
        "base_gdp": base_gdp,
        "base_pop": base_pop,
        "base_energy": total_final,
        "capital_stock": klem.capital_stock,
        "labor": klem.labor,
        "tfp": klem.tfp,
        "alpha": klem.alpha,
    })

    # Checks
    k_check = abs(klem.capital_stock - base_gdp * CAPITAL_OUTPUT_RATIO) < 1e-10
    l_check = abs(klem.labor - base_pop * LABOR_FORCE_PARTICIPATION) < 1e-10
    y_roundtrip = klem.tfp * klem.capital_stock ** klem.alpha * klem.labor ** (1.0 - klem.alpha)
    y_check = abs(y_roundtrip - base_gdp) / base_gdp < 1e-10 if base_gdp > 0 else True
    pos_check = klem.capital_stock > 0 and klem.labor > 0 and klem.tfp > 0

    init_checks.append({
        "region": region,
        "K = GDP * K/Y": "PASS" if k_check else "FAIL",
        "L = Pop * LFP": "PASS" if l_check else "FAIL",
        "A*K^a*L^(1-a) = GDP": "PASS" if y_check else f"FAIL ({y_roundtrip:.2f} vs {base_gdp:.2f})",
        "All positive": "PASS" if pos_check else "FAIL",
    })

init_df = pd.DataFrame(init_rows).set_index("region")
print("KLEM Initialization Values:")
display(init_df.round(4))

print("\nInitialization Checks:")
checks_df = pd.DataFrame(init_checks).set_index("region")
display(checks_df)

---
## 4. TFP Trajectory & Historical K Back-Solve

In [ ]:
# Build KLEM drivers and compute TFP trajectories
tfp_data = {}  # region -> {year: tfp}
klem_drivers = {}  # keep for later use

for region in R10_REGIONS:
    base_gdp = float(gdp_df.loc[region, BASE_YEAR])
    base_pop = float(pop_df.loc[region, BASE_YEAR])
    fd = DEFAULT_FINAL_DEMAND.get(region, {"industry": 5.0, "buildings": 5.0, "transport": 5.0})
    total_final = sum(fd.values())

    klem = KLEMDriver(base_gdp, base_pop, total_final)

    # Build SSP series
    ssp_gdp_series = {y: float(gdp_df.loc[region, y]) for y in MODEL_YEARS if y in gdp_df.columns}
    pop_series = {y: float(pop_df.loc[region, y]) for y in MODEL_YEARS if y in pop_df.columns}

    klem.init_tfp_trajectory(ssp_gdp_series, pop_series)
    tfp_data[region] = dict(klem._tfp_trajectory)
    klem_drivers[region] = klem

# TFP table (selected years)
tfp_display_years = [2000, 2005, 2010, 2015, 2020, 2030, 2050, 2070, 2100]
tfp_display_years = [y for y in tfp_display_years if y in MODEL_YEARS]

tfp_table = pd.DataFrame(
    {region: {y: tfp_data[region].get(y, np.nan) for y in tfp_display_years}
     for region in R10_REGIONS}
).T

print("TFP Trajectory (selected years):")
display(tfp_table.round(6))

In [ ]:
# TFP line plot
fig, ax = plt.subplots(figsize=(14, 6))

for region in R10_REGIONS:
    years = sorted(tfp_data[region].keys())
    values = [tfp_data[region][y] for y in years]
    ax.plot(years, values, label=region, marker=".", markersize=3)

ax.axvline(BASE_YEAR, color="gray", linestyle="--", alpha=0.5, label=f"Base year ({BASE_YEAR})")
ax.set_title("TFP Trajectories by Region")
ax.set_xlabel("Year")
ax.set_ylabel("TFP (A)")
ax.legend(fontsize=7, loc="upper left")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# TFP trajectory checks
tfp_checks = []

for region in R10_REGIONS:
    klem = klem_drivers[region]
    traj = tfp_data[region]

    # Check 1: TFP(BASE_YEAR) from trajectory matches init TFP
    init_tfp = klem.base_gdp / (
        (klem.base_gdp * CAPITAL_OUTPUT_RATIO) ** klem.alpha
        * (klem.base_population * LABOR_FORCE_PARTICIPATION) ** (1.0 - klem.alpha)
    )
    traj_base = traj.get(BASE_YEAR, np.nan)
    match = abs(traj_base - init_tfp) / init_tfp < 1e-10 if init_tfp > 0 else False

    # Check 2: All positive
    all_pos = all(v > 0 for v in traj.values())

    # Check 3: Historical TFP > base-year TFP (smaller K -> higher A)
    hist_years = [y for y in HISTORICAL_YEARS if y < BASE_YEAR and y in traj]
    hist_higher = all(traj[y] >= traj_base * 0.9 for y in hist_years)  # allow 10% tolerance

    # Check 4: TFP trend smooth (no jumps > 50% between periods)
    sorted_years = sorted(traj.keys())
    smooth = True
    for i in range(1, len(sorted_years)):
        ratio = traj[sorted_years[i]] / traj[sorted_years[i-1]] if traj[sorted_years[i-1]] > 0 else 1.0
        if ratio > 1.5 or ratio < 0.5:
            smooth = False
            break

    tfp_checks.append({
        "region": region,
        "Base TFP match": "PASS" if match else f"FAIL ({traj_base:.6f} vs {init_tfp:.6f})",
        "All positive": "PASS" if all_pos else "FAIL",
        "Historical >= 0.9*base": "PASS" if hist_higher else "FAIL",
        "Smooth trajectory": "PASS" if smooth else "FAIL",
    })

print("TFP Trajectory Checks:")
display(pd.DataFrame(tfp_checks).set_index("region"))

---
## 5. Single-Period KLEM Walkthrough

Step through the KLEM computation for **North America** at the base year (2020), then year 2025.

In [ ]:
region = "North America"
base_gdp = float(gdp_df.loc[region, BASE_YEAR])
base_pop = float(pop_df.loc[region, BASE_YEAR])
fd = DEFAULT_FINAL_DEMAND[region]
total_final = sum(fd.values())

# Fresh KLEM driver
klem = KLEMDriver(base_gdp, base_pop, total_final)
ssp_gdp_series = {y: float(gdp_df.loc[region, y]) for y in MODEL_YEARS if y in gdp_df.columns}
pop_series = {y: float(pop_df.loc[region, y]) for y in MODEL_YEARS if y in pop_df.columns}
klem.init_tfp_trajectory(ssp_gdp_series, pop_series)

print(f"=== Single-Period Walkthrough: {region} ===")
print(f"\nBase inputs: GDP={base_gdp:.1f} B$, Pop={base_pop:.1f} M, Energy={total_final:.1f} EJ")
print(f"Initial K={klem.capital_stock:.1f} B$, L={klem.labor:.1f} M, TFP={klem.tfp:.6f}")

# --- Year 2020 ---
print(f"\n--- Year {BASE_YEAR} ---")

# Step 1: Set TFP
klem.set_tfp_for_year(BASE_YEAR)
print(f"1. set_tfp_for_year({BASE_YEAR}) -> TFP = {klem.tfp:.6f}")

# Step 2: Gross output
Y = klem.compute_gross_output(base_pop)
print(f"2. compute_gross_output({base_pop:.1f}) -> Y = {Y:.2f} B$")
print(f"   Check: Y vs SSP GDP: {Y:.2f} vs {base_gdp:.2f} (diff = {abs(Y - base_gdp):.2e})")

# Step 3: Energy demand
E = klem.compute_energy_demand(Y, energy_price_index=1.0)
print(f"3. compute_energy_demand(Y, price_idx=1.0) -> E = {E:.4f} EJ")
print(f"   Check: E vs base_energy: {E:.4f} vs {total_final:.4f} (diff = {abs(E - total_final):.2e})")

# Step 4: Energy cost
avg_price = 5.0  # approximate $/GJ
cost = KLEMDriver.compute_energy_cost(E, avg_price)
print(f"4. compute_energy_cost({E:.2f}, {avg_price}) -> cost = {cost:.2f} B$")

# Step 5: Net output
net_Y = KLEMDriver.compute_net_output(Y, cost)
print(f"5. compute_net_output({Y:.2f}, {cost:.2f}) -> net_Y = {net_Y:.2f} B$")

# Step 6: Investment
I = klem.compute_investment(net_Y)
print(f"6. compute_investment({net_Y:.2f}) -> I = {I:.2f} B$/yr")
print(f"   s*net_Y = {SAVINGS_RATE * net_Y:.2f}, cap*K = {INVESTMENT_CAP_RATE * klem.capital_stock:.2f}")

# Step 7: Update capital
K_before = klem.capital_stock
klem.update_capital(I)
print(f"7. update_capital({I:.2f}) -> K: {K_before:.2f} -> {klem.capital_stock:.2f} B$")

In [ ]:
# --- Year 2025 ---
year_2025 = BASE_YEAR + TIMESTEP
pop_2025 = pop_series.get(year_2025, base_pop)

print(f"--- Year {year_2025} ---")
print(f"Capital carried from {BASE_YEAR}: K = {klem.capital_stock:.2f} B$")

klem.set_tfp_for_year(year_2025)
print(f"1. set_tfp_for_year({year_2025}) -> TFP = {klem.tfp:.6f}")

Y2 = klem.compute_gross_output(pop_2025)
ssp_gdp_2025 = ssp_gdp_series.get(year_2025, base_gdp)
print(f"2. compute_gross_output({pop_2025:.1f}) -> Y = {Y2:.2f} B$")
print(f"   SSP GDP({year_2025}) = {ssp_gdp_2025:.2f} B$")

E2 = klem.compute_energy_demand(Y2, 1.0)
print(f"3. Energy demand = {E2:.4f} EJ (scaled from {total_final:.1f} by GDP ratio {Y2/base_gdp:.4f})")

cost2 = KLEMDriver.compute_energy_cost(E2, avg_price)
net_Y2 = KLEMDriver.compute_net_output(Y2, cost2)
I2 = klem.compute_investment(net_Y2)
print(f"4. Cost={cost2:.2f}, Net output={net_Y2:.2f}, Investment={I2:.2f} B$/yr")

K_before2 = klem.capital_stock
klem.update_capital(I2)
print(f"5. K: {K_before2:.2f} -> {klem.capital_stock:.2f} B$ (delta = {klem.capital_stock - K_before2:.2f})")

---
## 6. Full Model Run & Results Comparison

In [ ]:
# Run the full model
results = run_model(ssp_data)
df = results_to_dataframe(results)

print(f"Results: {len(results)} records ({df['region'].nunique()} regions x {df['year'].nunique()} periods)")
print(f"Columns: {list(df.columns[:15])}...")

In [ ]:
# GDP Fidelity: scatter plot of gross_output vs ssp_reference_gdp
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Scatter: model GDP vs SSP GDP
ax = axes[0]
for region in R10_REGIONS:
    rdf = df[df["region"] == region]
    ax.scatter(rdf["ssp_reference_gdp_billion_usd"], rdf["gross_output_billion_usd"],
              s=10, alpha=0.7, label=region)

max_val = max(df["ssp_reference_gdp_billion_usd"].max(), df["gross_output_billion_usd"].max())
ax.plot([0, max_val], [0, max_val], "k--", alpha=0.5, label="1:1 line")
ax.set_xlabel("SSP Reference GDP (B$)")
ax.set_ylabel("Model Gross Output (B$)")
ax.set_title("GDP Fidelity: Model vs SSP")
ax.legend(fontsize=7, loc="upper left")
ax.grid(True, alpha=0.3)

# Compute fit statistics
valid = df[df["ssp_reference_gdp_billion_usd"] > 0].copy()
ss_res = ((valid["gross_output_billion_usd"] - valid["ssp_reference_gdp_billion_usd"]) ** 2).sum()
ss_tot = ((valid["ssp_reference_gdp_billion_usd"] - valid["ssp_reference_gdp_billion_usd"].mean()) ** 2).sum()
r2 = 1 - ss_res / ss_tot if ss_tot > 0 else np.nan
rmse = np.sqrt(ss_res / len(valid))
valid["rel_dev"] = abs(valid["gross_output_billion_usd"] - valid["ssp_reference_gdp_billion_usd"]) / valid["ssp_reference_gdp_billion_usd"]
max_dev = valid["rel_dev"].max()

# Deviation histogram
ax2 = axes[1]
ax2.hist(valid["rel_dev"] * 100, bins=50, color="steelblue", edgecolor="white")
ax2.set_xlabel("Relative Deviation (%)")
ax2.set_ylabel("Count")
ax2.set_title(f"GDP Deviation Distribution (R\u00b2={r2:.6f}, RMSE={rmse:.1f} B$, max={max_dev*100:.2f}%)")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"GDP Fidelity Stats: R\u00b2 = {r2:.6f}, RMSE = {rmse:.1f} B$, Max deviation = {max_dev*100:.2f}%")

In [ ]:
# Capital stock, energy demand, and emissions trajectories
fig, axes = plt.subplots(1, 3, figsize=(20, 6))

for region in R10_REGIONS:
    rdf = df[df["region"] == region].sort_values("year")
    axes[0].plot(rdf["year"], rdf["capital_stock_billion_usd"], label=region)
    axes[1].plot(rdf["year"], rdf["total_energy_ej"], label=region)
    axes[2].plot(rdf["year"], rdf["emissions_mtco2"], label=region)

axes[0].set_title("Capital Stock (B$)")
axes[0].set_xlabel("Year")
axes[0].grid(True, alpha=0.3)

axes[1].set_title("Total Energy Demand (EJ)")
axes[1].set_xlabel("Year")
axes[1].grid(True, alpha=0.3)

axes[2].set_title("CO2 Emissions (MtCO2)")
axes[2].set_xlabel("Year")
axes[2].grid(True, alpha=0.3)
axes[2].legend(fontsize=7, bbox_to_anchor=(1.02, 1), loc="upper left")

plt.tight_layout()
plt.show()

In [ ]:
# Base-year summary table
base_df = df[df["year"] == BASE_YEAR].set_index("region").reindex(R10_REGIONS)

summary_cols = [
    "gross_output_billion_usd", "ssp_reference_gdp_billion_usd",
    "net_output_billion_usd", "capital_stock_billion_usd",
    "investment_billion_usd", "energy_cost_billion_usd",
    "tfp", "total_energy_ej", "emissions_mtco2",
]

print(f"\nBase Year ({BASE_YEAR}) KLEM Outputs:")
display(base_df[summary_cols].round(2))

---
## 7. Diagnostic Checks & Summary

Automated pass/fail checks across all regions and years.

In [ ]:
check_names = [
    "GDP reproduction (<30%)",
    "Capital positivity",
    "TFP positivity",
    "Energy positivity",
    "Net output bound",
    "Investment bound",
    "Capital accumulation",
    "Historical K consistency",
]

summary_results = {check: {} for check in check_names}

for region in R10_REGIONS:
    rdf = df[df["region"] == region].sort_values("year").copy()

    # 1. GDP reproduction: |gross_output - ssp_gdp| / ssp_gdp < 30%
    # Note: deviations are expected because energy cost feedback alters capital
    # accumulation relative to the TFP calibration path. The TFP is calibrated
    # assuming reference K evolution, but solver energy costs change K.
    valid_rows = rdf[rdf["ssp_reference_gdp_billion_usd"] > 0]
    rel_err = abs(valid_rows["gross_output_billion_usd"] - valid_rows["ssp_reference_gdp_billion_usd"]) / valid_rows["ssp_reference_gdp_billion_usd"]
    max_err = rel_err.max() * 100
    summary_results["GDP reproduction (<30%)"][region] = "PASS" if (rel_err < 0.30).all() else f"FAIL (max {max_err:.1f}%)"

    # 2. Capital positivity
    summary_results["Capital positivity"][region] = "PASS" if (rdf["capital_stock_billion_usd"] > 0).all() else "FAIL"

    # 3. TFP positivity
    summary_results["TFP positivity"][region] = "PASS" if (rdf["tfp"] > 0).all() else "FAIL"

    # 4. Energy positivity: check only from BASE_YEAR onward (historical periods
    # may have zero energy demand because solve_period computes demand from the
    # full energy system which may not be initialized for historical years)
    future_rdf = rdf[rdf["year"] >= BASE_YEAR]
    summary_results["Energy positivity"][region] = "PASS" if (future_rdf["total_energy_ej"] > 0).all() else "FAIL"

    # 5. Net output bound: 1% of gross <= net <= gross
    lower_ok = (rdf["net_output_billion_usd"] >= 0.01 * rdf["gross_output_billion_usd"] - 1e-6).all()
    upper_ok = (rdf["net_output_billion_usd"] <= rdf["gross_output_billion_usd"] + 1e-6).all()
    summary_results["Net output bound"][region] = "PASS" if (lower_ok and upper_ok) else "FAIL"

    # 6. Investment bound: I <= min(s*net_Y, cap_rate*K)
    inv_limit = np.minimum(
        SAVINGS_RATE * rdf["net_output_billion_usd"],
        INVESTMENT_CAP_RATE * rdf["capital_stock_billion_usd"]
    )
    summary_results["Investment bound"][region] = "PASS" if (rdf["investment_billion_usd"] <= inv_limit + 1e-6).all() else "FAIL"

    # 7. Capital accumulation: K grows over time for positive savings
    k_vals = rdf["capital_stock_billion_usd"].values
    summary_results["Capital accumulation"][region] = "PASS" if k_vals[-1] > k_vals[0] else "FAIL"

    # 8. Historical K consistency: TFP at 2000 >= TFP at 2020
    # With backward K solve, historical K < base K, so TFP >= base TFP.
    # Equal is valid when GDP growth exactly matches capital accumulation.
    traj = tfp_data.get(region, {})
    if 2000 in traj and BASE_YEAR in traj:
        summary_results["Historical K consistency"][region] = (
            "PASS" if traj[2000] >= traj[BASE_YEAR] - 1e-10 else
            f"FAIL ({traj[2000]:.4f} < {traj[BASE_YEAR]:.4f})"
        )
    else:
        summary_results["Historical K consistency"][region] = "N/A"

# Build summary table
summary_table = pd.DataFrame(summary_results)

print("=" * 80)
print("DIAGNOSTIC CHECK SUMMARY")
print("=" * 80)
display(summary_table)

# Count pass/fail
total_checks = summary_table.size
pass_count = (summary_table == "PASS").sum().sum()
fail_count = summary_table.apply(lambda col: col.str.startswith("FAIL")).sum().sum()
na_count = (summary_table == "N/A").sum().sum()

print(f"\nTotal: {total_checks} checks | PASS: {pass_count} | FAIL: {fail_count} | N/A: {na_count}")
if fail_count == 0:
    print("All checks PASSED.")
else:
    print(f"WARNING: {fail_count} check(s) FAILED. See table above for details.")

# Additional: show GDP deviation statistics
print("\n--- GDP Deviation Details (energy cost feedback) ---")
for region in R10_REGIONS:
    rdf = df[(df["region"] == region) & (df["ssp_reference_gdp_billion_usd"] > 0)]
    rel = abs(rdf["gross_output_billion_usd"] - rdf["ssp_reference_gdp_billion_usd"]) / rdf["ssp_reference_gdp_billion_usd"]
    print(f"  {region:45s}: mean={rel.mean()*100:5.1f}%, max={rel.max()*100:5.1f}%")